In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
import joblib

In [2]:
# 1. Load the Adult Income Dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['Age', 'Workclass', 'fnlwgt', 'Education', 'Education-Num', 'Marital-Status', 
           'Occupation', 'Relationship', 'Race', 'Sex', 'Capital-Gain', 'Capital-Loss', 
           'Hours-per-week', 'Country', 'Target']
df = pd.read_csv(url, names=columns, na_values=" ?", skipinitialspace=True)

In [3]:
df.head()

,Age,Workclass,fnlwgt,Education,Education-Num,Marital-Status,Occupation,Relationship,Race,Sex,Capital-Gain,Capital-Loss,Hours-per-week,Country,Target
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
df.shape

(32561, 15)

In [5]:
df.isnull().sum()

Age               0
Workclass         0
fnlwgt            0
Education         0
Education-Num     0
Marital-Status    0
Occupation        0
Relationship      0
Race              0
Sex               0
Capital-Gain      0
Capital-Loss      0
Hours-per-week    0
Country           0
Target            0
dtype: int64

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Age             32561 non-null  int64
 1   Workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   Education       32561 non-null  str  
 4   Education-Num   32561 non-null  int64
 5   Marital-Status  32561 non-null  str  
 6   Occupation      32561 non-null  str  
 7   Relationship    32561 non-null  str  
 8   Race            32561 non-null  str  
 9   Sex             32561 non-null  str  
 10  Capital-Gain    32561 non-null  int64
 11  Capital-Loss    32561 non-null  int64
 12  Hours-per-week  32561 non-null  int64
 13  Country         32561 non-null  str  
 14  Target          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 6.2 MB


In [7]:
# 2. Preprocessing
# Drop missing values as precationary step, even though data is already cleaned
df.dropna(inplace=True)

In [8]:
# Encode categorical variables
# This is a simple encoding method.
le = LabelEncoder()
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

In [9]:
df.head()

,Age,Workclass,fnlwgt,Education,Education-Num,Marital-Status,Occupation,Relationship,Race,Sex,Capital-Gain,Capital-Loss,Hours-per-week,Country,Target
0,39,7,77516,9,13,4,1,1,4,1,2174,0,40,39,0
1,50,6,83311,9,13,2,4,0,4,1,0,0,13,39,0
2,38,4,215646,11,9,0,6,1,4,1,0,0,40,39,0
3,53,4,234721,1,7,2,6,0,2,1,0,0,40,39,0
4,28,4,338409,9,13,2,10,5,2,0,0,0,40,5,0


In [10]:
# Split the dataset into features (X) and target (y)
data = df.copy()
X = data.iloc[:, :-1]  # all columns except the last
y = data.iloc[:, -1]   # Target column

In [11]:
# 3. Data Splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# Scale the features (Crucial for KNN and SVM)
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

In [13]:
# 4. Save test data for Streamlit (Combining X_test and y_test)
test_data = X_test.copy()
test_data['Target'] = y_test.values
test_data.to_csv("test_data.csv", index=False)
print("test_data.csv saved successfully.\n")

test_data.csv saved successfully.



In [14]:
# Setup for model saving
os.makedirs("model", exist_ok=True)

In [15]:
# Helper function to evaluate and print metrics
def evaluate_model(model_name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    # Formatted row for the README comparison table
    print(f"{model_name:<25} | {acc:<8.4f} | {auc:<8.4f} | {prec:<9.4f} | {rec:<8.4f} | {f1:<8.4f} | {mcc:<8.4f}")

In [16]:
# Print Table Header
print(f"{'ML Model Name':<25} | {'Accuracy':<8} | {'AUC':<8} | {'Precision':<9} | {'Recall':<8} | {'F1':<8} | {'MCC':<8}")
print("-" * 95)

# --- Model 1: Logistic Regression ---
m1 = LogisticRegression()
m1.fit(X_train, y_train)
ypred = m1.predict(X_test)
yprob1 = m1.predict_proba(X_test)[:, 1]
evaluate_model("Logistic Regression", y_test, ypred, yprob1)
joblib.dump(m1, "model/logistic_regression.pkl")

# --- Model 2: Decision Tree ---
treeclassifier = DecisionTreeClassifier(random_state=42)
treeclassifier.fit(X_train, y_train)
ypred2 = treeclassifier.predict(X_test)
yprob2 = treeclassifier.predict_proba(X_test)[:, 1]
evaluate_model("Decision Tree", y_test, ypred2, yprob2)
joblib.dump(treeclassifier, "model/decision_tree.pkl")

# --- Model 3: K-Nearest Neighbors ---
knn = KNeighborsClassifier(n_neighbors=5, algorithm='auto')
knn.fit(X_train, y_train)
ypred3 = knn.predict(X_test)
yprob3 = knn.predict_proba(X_test)[:, 1]
evaluate_model("KNN", y_test, ypred3, yprob3)
joblib.dump(knn, "model/knn.pkl")

# --- Model 4: Naive Bayes ---
gnb = GaussianNB()
gnb.fit(X_train, y_train)
ypred4 = gnb.predict(X_test)
yprob4 = gnb.predict_proba(X_test)[:, 1]
evaluate_model("Naive Bayes", y_test, ypred4, yprob4)
joblib.dump(gnb, "model/naive_bayes.pkl")

# --- Model 5: Support Vector Machine ---
# Note: probability=True is required to calculate the AUC score for SVM
svc = SVC(kernel='linear', probability=True, random_state=42)
svc.fit(X_train, y_train)
ypred5 = svc.predict(X_test)
yprob5 = svc.predict_proba(X_test)[:, 1]
evaluate_model("Support Vector Machine", y_test, ypred5, yprob5)
joblib.dump(svc, "model/svm.pkl")

# Save the scaler so the Streamlit app can scale uploaded user data properly
joblib.dump(scaler, "model/scaler.pkl")
print("\nAll models and artifacts saved to the 'model/' directory.")

ML Model Name             | Accuracy | AUC      | Precision | Recall   | F1       | MCC     
-----------------------------------------------------------------------------------------------
Logistic Regression       | 0.8247   | 0.8568   | 0.7113    | 0.4596   | 0.5584   | 0.4722  
Decision Tree             | 0.8131   | 0.7516   | 0.6083    | 0.6327   | 0.6203   | 0.4966  
KNN                       | 0.8353   | 0.8572   | 0.6786    | 0.6022   | 0.6381   | 0.5335  
Naive Bayes               | 0.8093   | 0.8613   | 0.7064    | 0.3584   | 0.4755   | 0.4060  
Support Vector Machine    | 0.8174   | 0.8518   | 0.8003    | 0.3240   | 0.4613   | 0.4299  

All models and artifacts saved to the 'model/' directory.
